In [ ]:
# 1. Solve the N-Queens problem using Hill climbing:
# ● Start with a random state
# ● At each step, compute cost of all neighbors and dislay the best
# ● Display the final result (positions on board in 2-D)

# 2. Solve the N-Queens problem using Simulated Annealing:
# ● Start with a random state
# ● Initialize temperature T
# ● At each step, show current cost, neighbor cost and whether it is
# accepted/rejected
# ● Display the final result (positions on board in 2-D)

In [ ]:
import random

def cost(board):
    n = len(board)
    attacks = 0
    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j] or abs(board[i] - board[j]) == abs(i - j):
                attacks += 1
    return attacks

def get_neighbors(board):
    n = len(board)
    neighbors = []
    for col in range(n):
        for row in range(n):
            if row != board[col]:
                new_board = board[:]
                old_row = new_board[col]
                new_board[col] = row
                neighbors.append((new_board, col, old_row, row, cost(new_board)))

    return neighbors

def print_board(board):
    n = len(board)

    print("\nFinal board (2-D):")
    for row in range(n):
        for col in range(n):
            if board[col] == row:
                print("Q", end=" ")
            else:
                print(".", end=" ")
        print()

    print("\nQueen positions (row, col) in 1-based indexing:")
    positions = []
    for col in range(n):
        positions.append((board[col] + 1, col + 1))
    print(positions)

def hill_climbing(n, seed=None):
    if seed is not None:
        random.seed(seed)

    board = [random.randint(0, n - 1) for _ in range(n)]

    print("Random initial state (row for each column, 1-based):", [x + 1 for x in board])
    print("Initial cost =", cost(board))
    print()

    step = 1

    while True:
        current_cost = cost(board)

        if current_cost == 0:
            print("Solution found.\n")
            break

        neighbors = get_neighbors(board)
        best = min(neighbors, key=lambda x: x[4])

        best_board = best[0]
        col = best[1]
        old_row = best[2]
        new_row = best[3]
        best_cost = best[4]

        print("Step", step)
        print("Current state =", [x + 1 for x in board], ", cost =", current_cost)
        print("Best neighbor =", [x + 1 for x in best_board], ", cost =", best_cost)
        print("Move queen in column", col + 1, "from row", old_row + 1, "to row", new_row + 1)
        print()

        if best_cost >= current_cost:
            print("Stopped at local minimum / plateau.\n")
            break

        board = best_board
        step += 1

    print_board(board)
    print("\nFinal cost =", cost(board))

n = 8
hill_climbing(n, seed=42)

Random initial state (row for each column, 1-based): [2, 1, 5, 4, 4, 3, 2, 2]
Initial cost = 10

Step 1:
Current state = [2, 1, 5, 4, 4, 3, 2, 2], cost = 10
Best neighbor = [2, 1, 5, 4, 8, 3, 2, 2], cost = 6
Move queen in column 5 from row 4 to row 8

Step 2:
Current state = [2, 1, 5, 4, 8, 3, 2, 2], cost = 6
Best neighbor = [6, 1, 5, 4, 8, 3, 2, 2], cost = 3
Move queen in column 1 from row 2 to row 6

Step 3:
Current state = [6, 1, 5, 4, 8, 3, 2, 2], cost = 3
Best neighbor = [6, 1, 5, 4, 8, 3, 5, 2], cost = 2
Move queen in column 7 from row 2 to row 5

Step 4:
Current state = [6, 1, 5, 4, 8, 3, 5, 2], cost = 2
Best neighbor = [6, 1, 7, 4, 8, 3, 5, 2], cost = 1
Move queen in column 3 from row 5 to row 7

Step 5:
Current state = [6, 1, 7, 4, 8, 3, 5, 2], cost = 1
Best neighbor = [6, 1, 7, 4, 8, 3, 5, 3], cost = 1
Move queen in column 8 from row 2 to row 3

Stopped at local minimum / plateau.


Final board (2-D):
. Q . . . . . .
. . . . . . . Q
. . . . . Q . .
. . . Q . . . .
. . . . . .

In [ ]:
import random
import math

def cost(state):
    n = len(state)
    attacks = 0
    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1, r2 = state[c1], state[c2]
            if r1 == r2 or abs(r1 - r2) == abs(c1 - c2):
                attacks += 1
    return attacks

def random_neighbor(state):
    n = len(state)
    new_state = state[:]
    col = random.randint(0, n - 1)
    new_row = random.randint(0, n - 1)
    while new_row == new_state[col]:
        new_row = random.randint(0, n - 1)
    old_row = new_state[col]
    new_state[col] = new_row
    return new_state, col, old_row, new_row

def print_board(state):
    n = len(state)
    print("\nFinal board (2-D):")
    for r in range(n):
        row = []
        for c in range(n):
            if state[c] == r:
                row.append("Q")
            else:
                row.append(".")
        print(" ".join(row))

    print("\nQueen positions (row, col) in 1-based indexing:")
    positions = []
    for c in range(n):
        positions.append((state[c] + 1, c + 1))
    print(positions)

def simulated_annealing_n_queens(n, T=100.0, cooling=0.95, min_T=0.001, max_steps=200, seed=None):
    if seed is not None:
        random.seed(seed)

    current = [random.randint(0, n - 1) for _ in range(n)]
    current_cost = cost(current)

    print(f"Initial random state (row for each column, 1-based): {[x + 1 for x in current]}")
    print(f"Initial cost = {current_cost}")
    print(f"Initial temperature = {T}\n")

    step = 0

    while T > min_T and step < max_steps:
        if current_cost == 0:
            print("Solution found.\n")
            break

        neighbor, col, old_row, new_row = random_neighbor(current)
        neighbor_cost = cost(neighbor)
        delta = neighbor_cost - current_cost

        accepted = False
        reason = ""

        if delta < 0:
            accepted = True
            reason = "better state"
        else:
            prob = math.exp(-delta / T)
            rand_val = random.random()
            if rand_val < prob:
                accepted = True
                reason = f"accepted probabilistically (p={prob:.4f})"
            else:
                reason = f"rejected probabilistically (p={prob:.4f})"

        print(f"Step {step + 1}:")
        print(f"Temperature = {T:.4f}")
        print(f"Current state = {[x + 1 for x in current]}, cost = {current_cost}")
        print(f"Neighbor state = {[x + 1 for x in neighbor]}, cost = {neighbor_cost}")
        print(f"Move queen in column {col + 1} from row {old_row + 1} to row {new_row + 1}")
        print(f"Decision = {'ACCEPTED' if accepted else 'REJECTED'} ({reason})\n")

        if accepted:
            current = neighbor
            current_cost = neighbor_cost

        T *= cooling
        step += 1

    print("Final state:")
    print(f"State (row for each column, 1-based): {[x + 1 for x in current]}")
    print(f"Final cost = {current_cost}")
    print_board(current)

# Example input
n = 8
simulated_annealing_n_queens(n, T=100.0, cooling=0.95, min_T=0.001, max_steps=200, seed=42)

Initial random state (row for each column, 1-based): [2, 1, 5, 4, 4, 3, 2, 2]
Initial cost = 10
Initial temperature = 100.0

Step 1:
Temperature = 100.0000
Current state = [2, 1, 5, 4, 4, 3, 2, 2], cost = 10
Neighbor state = [2, 1, 5, 4, 4, 3, 1, 2], cost = 10
Move queen in column 7 from row 2 to row 1
Decision = ACCEPTED (accepted probabilistically (p=1.0000))

Step 2:
Temperature = 95.0000
Current state = [2, 1, 5, 4, 4, 3, 1, 2], cost = 10
Neighbor state = [2, 1, 5, 1, 4, 3, 1, 2], cost = 10
Move queen in column 4 from row 4 to row 1
Decision = ACCEPTED (accepted probabilistically (p=1.0000))

Step 3:
Temperature = 90.2500
Current state = [2, 1, 5, 1, 4, 3, 1, 2], cost = 10
Neighbor state = [2, 1, 5, 1, 4, 3, 4, 2], cost = 9
Move queen in column 7 from row 1 to row 4
Decision = ACCEPTED (better state)

Step 4:
Temperature = 85.7375
Current state = [2, 1, 5, 1, 4, 3, 4, 2], cost = 9
Neighbor state = [2, 1, 5, 1, 4, 3, 4, 5], cost = 12
Move queen in column 8 from row 2 to row 5
Decisi